In [ ]:
from plot_umap import *
import pandas as pd

## Load data

In [ ]:
hs_train = pd.read_json("../../experiment/datasets/goldenswag/original/golden_swag.json")

random_state = 1111

In [ ]:
# Encode context and endings
options = [0, 1, 2, 3]

data = {}
for id, entry in hs_train.iterrows():
    context = entry["ctx"]
    correct_ending = entry["endings"][entry["label"]]
    wrong_endings = [entry["endings"][i] for i in options if i != entry["label"]]

    data_point = {
        "text": context + " " + correct_ending,
        "correct": 1,
        "category": entry["activity_label"],
        "task_id": id
    }
    data[len(data)] = data_point
    for wrong in wrong_endings:
        data_point = {
            "text": context + " " + wrong,
            "correct": 0,
            "category": entry["activity_label"],
            "task_id": id
        }
        data[len(data)] = data_point

df = pd.DataFrame.from_dict(data, orient="index")
print("Total samples:", len(df))

In [ ]:
def filter_df(df: DataFrame, top_n_categories: int, n_samples: int) -> DataFrame:
    # Get top N categories
    top_n_categories = df['category'].value_counts().head(top_n_categories).index.tolist()
    filtered_df = df[df["category"].isin(top_n_categories)]

    # Group by task_id and sample groups rather than individual rows
    grouped = filtered_df.groupby('task_id')

    # Get list of all task_ids
    task_ids = list(grouped.groups.keys())

    # Sample task_ids (not individual rows)
    sampled_task_ids = pd.Series(task_ids).sample(n=min(round(n_samples / 4), len(task_ids)), random_state=random_state)

    # Get all rows for the sampled task_ids
    sampled_df = filtered_df[filtered_df['task_id'].isin(sampled_task_ids)]

    print(f"Number of samples={len(sampled_df)}, got {len(sampled_task_ids)} task groups")
    return sampled_df.reset_index(drop=True)


def get_random_sampled(df: DataFrame, n_samples: int) -> DataFrame:
    # Group by task_id to ensure we sample complete task groups
    grouped = df.groupby('task_id')

    # Get list of all task_ids
    task_ids = list(grouped.groups.keys())

    # Calculate how many task groups we need to sample
    # Each task group typically contains 4 rows (context + 3 endings)
    # So we divide the requested number of samples by 4
    n_task_groups = min(round(n_samples / 4), len(task_ids))

    # Randomly sample task_ids
    sampled_task_ids = pd.Series(task_ids).sample(n=n_task_groups, random_state=random_state)

    # Get all rows for the sampled task_ids
    sampled_df = df[df['task_id'].isin(sampled_task_ids)]

    print(f"Requested {n_samples} samples, got {len(sampled_df)} samples from {n_task_groups} task groups")

    return sampled_df.reset_index(drop=True)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
# Load model and move to GPU
model_finetuned = AutoModel.from_pretrained("../models/Electra_golden_swag_train/epoch35").to(device)
model = AutoModel.from_pretrained("google/electra-base-discriminator").to(device)
tokenizer = AutoTokenizer.from_pretrained("google/electra-base-discriminator")
model.train(False)
model_finetuned.train(False)


def get_embeddings(df: DataFrame, fine_tuned=False) -> Tensor:
    # Check if MPS is available

    inputs = tokenizer(df["text"].tolist(), padding=True, truncation=True, return_tensors="pt").to(
        device)  # Move all tensors to GPU

    # Generate embeddings in batches (faster and avoids OOM)
    batch_size = 64  # Adjust based on GPU memory
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(inputs["input_ids"]), batch_size):
            batch = {k: v[i:i + batch_size] for k, v in inputs.items()}
            if fine_tuned:
                batch_embeddings = model_finetuned(**batch, return_dict=True).last_hidden_state[:, -1, :]
            else:
                batch_embeddings = model(**batch, return_dict=True).last_hidden_state[:, -1, :]
            # batch_embeddings = model(**batch, return_dict=True).pooler_output
            embeddings.append(batch_embeddings.cpu())  # Move back to CPU

    # Concatenate all batches
    return torch.cat(embeddings, dim=0)

## Top 3 categories & 1'000 samples UMAP 3D

In [ ]:
df_1 = get_random_sampled(df, 500)
embeddings_1 = get_embeddings(df_1)
embeddings_1_finetuned = get_embeddings(df_1, fine_tuned=True)

In [ ]:
neighbors = list(range(10, 56, 2))
for neighbor in neighbors:
    plot_umap_3d_compared(embeddings_1, embeddings_1_finetuned, df_1, neighbor)

## Single category & 100 samples

In [ ]:
## Single category & 1'000 samples
df_top1_100 = filter_df(df, 1, 100)
embeddings_1_100 = get_embeddings(df_top1_100)
embeddings_1_100_finetuned = get_embeddings(df_top1_100, fine_tuned=True)

In [ ]:
neighbors = list(range(5, 30, 5))
for neighbor in neighbors:
    plot_umap_compared(embeddings_1_100, embeddings_1_100_finetuned, df_top1_100, neighbor, color_label="task_id")

## Top 10 categories & 100 samples

In [ ]:
df_top10_100 = filter_df(df, 10, 100)
embeddings_10_100 = get_embeddings(df_top10_100)
embeddings_10_100_finetuned = get_embeddings(df_top10_100, fine_tuned=True)

In [ ]:
neighbors = list(range(15, 35, 5))
for neighbor in neighbors:
    plot_umap_compared(embeddings_10_100, embeddings_10_100_finetuned, df_top10_100, neighbor)

Single Samples

## Single category & 100 samples

In [ ]:
df_top1_20 = filter_df(df, 1, 100)
embeddings_1_20 = get_embeddings(df_top1_20)
embeddings_1_20_finetuned = get_embeddings(df_top1_20, fine_tuned=True)

In [ ]:
neighbors = list(range(15, 35, 5))
for neighbor in neighbors:
    plot_umap_1d_compared(embeddings_1_20, embeddings_1_20_finetuned, df_top1_20, neighbor, color_label="task_id")

## Top 3 categories & 200 samples 3D UMAP

In [ ]:
df_y = get_random_sampled(df, 200)
embeddings_1 = get_embeddings(df_y)
embeddings_1_finetuned = get_embeddings(df_y, fine_tuned=True)

In [ ]:
plot_pca_3d_compared(embeddings_1, embeddings_1_finetuned, df_y)

## Pair of embeddings with PCA

In [ ]:
from plot_pca import *

def compare_pairs_with_pca_3d(
    df: pd.DataFrame,
    num_samples: int = 1000,
    num_splits: int = 2,
    samples_per_plot: int = 2,
    color_label: str = "task_id"
):
    """
    Automatically compare pairs of embeddings (regular vs. fine-tuned) using UMAP.

    Args:
        df: DataFrame containing the data.
        num_samples: Number of samples to randomly select from the DataFrame.
        num_splits: Number of splits to divide the embeddings into.
        samples_per_plot: Number of samples to include in each UMAP plot.
        color_label: Column name for coloring points in the UMAP plot.
    """
    # Get random samples
    df_sampled = get_random_sampled(df, num_samples)

    # Get embeddings
    embeddings = get_embeddings(df_sampled)
    embeddings_finetuned = get_embeddings(df_sampled, fine_tuned=True)

    # Split embeddings
    split_size = embeddings.size(0) // num_splits
    split_embeddings = torch.split(embeddings, split_size, dim=0)
    split_embeddings_finetuned = torch.split(embeddings_finetuned, split_size, dim=0)

    # Iterate over splits and plot
    for i in range(num_splits):
        start_idx = i * samples_per_plot
        end_idx = start_idx + samples_per_plot

        # Get the subset of the DataFrame for the current split
        df_subset = df_sampled.iloc[start_idx:end_idx]

        # Plot UMAP for the current split
        plot_pca_3d_compared(
            split_embeddings[i],
            split_embeddings_finetuned[i],
            df_subset,
            color_label=color_label
        )

In [ ]:
compare_pairs_with_pca_3d(df, num_samples=10, num_splits=4, samples_per_plot=2, color_label="task_id")